## FovealGAT: components overview

In [ ]:
import only_gaze_confs as conf

With the gaze patch features and encodings aggregated into graph nodes, we can feed it through our network. We'll go through each component.

Fetch sample batch

In [ ]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label)
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)

nodes = torch.cat([fixations, seqs, feats], 1).float()
nodes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

In [ ]:
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))

### GAT Layer
<a id="gat_layer"></a>

The first component is the Graph Attention Convolution Layer.

In [ ]:
from model.gat_conv import CausalGATConv

set number of attention heads for the layer

In [ ]:
att_heads = 4

In [ ]:
gcnn = CausalGATConv(ns.shape[0],
                     ns.shape[0],
                     att_heads,
                     device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

### XYTGATLayer

a gat layer that has attention mechanisms that look only into pair of nodex xy coord and pairs of time duration

In [ ]:
from xyt_gat_conv import CausalXYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)

In [ ]:
import torch

In [ ]:
gcnn = CausalXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [ ]:
gcnn

In [ ]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats))
conv_nodes.shape

### EuclideanTGATLayer

a gat layer that ads delta x, delta y and euclidean distance between two fixations to spatial attention adjacency matrix

In [ ]:
from xyt_gat_conv import CausalEuclideanTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

In [ ]:
gcnn = CausalEuclideanTGATConv(ns.shape[0],
                               ns.shape[0],
                               8,
                               device=conf.device)

In [ ]:
gcnn

In [ ]:
conv_nodes = gcnn((ns, splits, xy_eucs, t_mats))
conv_nodes.shape

### CausalSummaryTGATLayer

In [ ]:
from xyt_gat_conv import CausalSummaryXYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

In [ ]:
gcnn = CausalSummaryXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [ ]:
gcnn

test for first convolution without summary node

In [ ]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats))
conv_nodes.shape

test for other convolutions, with a summary node present already

In [ ]:
test = torch.cat([ns, torch.zeros((ns.shape[0], 3)).to(conf.device)], dim=-1)
test.shape

In [ ]:
conv_nodes = gcnn((test, splits, xy_mats, t_mats))
conv_nodes.shape

### CausalEuclideanTGATConv

In [ ]:
from xyt_gat_conv import CausalEuclideanSummaryTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

In [ ]:
gcnn = CausalEuclideanSummaryTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [ ]:
gcnn

test for first convolution without summary node

In [ ]:
conv_nodes = gcnn((ns, splits, xy_eucs, t_mats))
conv_nodes.shape

test for other convolutions, with a summary node present already

In [ ]:
test = torch.cat([ns, torch.zeros((ns.shape[0], 3)).to(conf.device)], dim=-1)
test.shape

In [ ]:
conv_nodes = gcnn((test, splits, xy_eucs, t_mats))
conv_nodes.shape

### CausalIOU_XYTGATConv

In [ ]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label, target_fixation_keys=('t', 'x', 'y', 'top_left', 'bottom_right'))
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)
fixations, bboxes = torch.split(fixations, (3, 4), dim=1)

nodes = torch.cat([fixations, seqs, feats], 1).float()
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))
bboxes = bboxes.squeeze(-1).squeeze(-1).transpose(1, 0)
ns.shape, bboxes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

In [ ]:
fixations.shape

In [ ]:
from iou_xyt_gat_conv import CausalIOU_XYTGATConv, CausalSummaryIOU_XYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_bbox_iou_encodings

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
iou_mats = get_bbox_iou_encodings(bboxes, splits)

In [ ]:
import torch

In [ ]:
gcnn = CausalSummaryIOU_XYTGATConv(ns.shape[0],
                            ns.shape[0],
                            8,
                            device=conf.device)

In [ ]:
gcnn

In [ ]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats, iou_mats))
conv_nodes.shape

In [ ]:
from iou_xyt_gat_conv import CausalIOU_EuclideanTGATConv, CausalEuclideanSummaryIOU_TGATConv

In [ ]:
gcnn = CausalEuclideanSummaryIOU_TGATConv(ns.shape[0],
                                   ns.shape[0],
                                   8,
                                   device=conf.device)

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

In [ ]:
gcnn((ns, splits, xy_eucs, t_mats, iou_mats)).shape

### GazeGrapher
<a id="grapher"></a>

GazeGrapher is a module that aggregates several GAT layers in sequence, performing all graph convolutions before graph readout and prediction.

PyTorch's native nn.Sequential does not keep residual tensors between its internal steps, so we need to extend it. Graph readout and predictions will need residual, intermediary, graphs

In [ ]:
from residual_sequential import MarkovResidualSequential, DenseResidualSequential

We need input sizes for each of the graphs GAT Layers, and the residual layers' indices and GAT conv class of choice

In [ ]:
layers = 4
gat_channels = 256 # input size and 4 GAT convolution layers
residual_gat_levels = [False, True, True, True]
att_heads = [16, 8, 8, 8]
output_gat_levels = None
gat_conv = CausalGATConv

In [ ]:
from grapher import GazeGrapher, XYTGazeGrapher

In [ ]:
from torch import nn
from torch_geometric.nn.norm import GraphNorm
import torch.nn.functional as F

from model.gat_conv import CausalGATConv

In [ ]:
grapher = GazeGrapher(layers=layers,
                      output_gat_levels=output_gat_levels,
                      residual_gat_levels=residual_gat_levels,
                      att_heads=att_heads,
                      gat_conv_args={'channels': gat_channels,
                                     'act_att': F.leaky_relu,
                                     'act_gat': F.elu,
                                     'norm': GraphNorm,
                                     'dropout': 0.1,
                                     'bias': True,
                                     'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                      device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
grapher = XYTGazeGrapher(layers=layers,
                         output_gat_levels=output_gat_levels,
                         residual_gat_levels=residual_gat_levels,
                         att_heads=att_heads,
                         gat_conv_args={'channels': gat_channels,
                                        'xy_heads_ratio': 0.2,
                                        't_heads_ratio': 0.1,
                                        'act_att': F.leaky_relu,
                                        'act_gat': F.elu,
                                        'norm': GraphNorm,
                                        'dropout': 0.1,
                                        'bias': True,
                                        'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                         device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
grapher = XYTGazeGrapher(gat_conv=CausalSummaryXYTGATConv,
                         layers=layers,
                         output_gat_levels=output_gat_levels,
                         residual_gat_levels=residual_gat_levels,
                         att_heads=att_heads,
                         gat_conv_args={'channels': gat_channels,
                                        'xy_heads_ratio': 0.2,
                                        't_heads_ratio': 0.1,
                                        'act_att': F.leaky_relu,
                                        'act_gat': F.elu,
                                        'norm': GraphNorm,
                                        'dropout': 0.1,
                                        'bias': True,
                                        'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                         device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
from grapher import EuclideanTGazeGrapher

In [ ]:
grapher = EuclideanTGazeGrapher(layers=layers,
                                output_gat_levels=output_gat_levels,
                                residual_gat_levels=residual_gat_levels,
                                att_heads=att_heads,
                                gat_conv_args={'channels': gat_channels,
                                               'xy_heads_ratio': 0.2,
                                               't_heads_ratio': 0.1,
                                               'act_att': F.leaky_relu,
                                               'act_gat': F.elu,
                                               'norm': GraphNorm,
                                               'dropout': 0.1,
                                               'bias': True,
                                               'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                                device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
graphs = grapher((ens, splits, fixations)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label, target_fixation_keys=('t', 'x', 'y', 'top_left', 'bottom_right'))
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)
fixations, bboxes = torch.split(fixations, (3, 4), dim=1)

nodes = torch.cat([fixations, seqs, feats], 1).float()
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))
bboxes = bboxes.squeeze(-1).squeeze(-1).transpose(1, 0)
ns.shape, bboxes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

In [ ]:
from grapher import IOU_XYTGazeGrapher

In [ ]:
grapher = IOU_XYTGazeGrapher(gat_conv=CausalSummaryIOU_XYTGATConv,
                                layers=layers,
                                output_gat_levels=output_gat_levels,
                                residual_gat_levels=residual_gat_levels,
                                att_heads=att_heads,
                                gat_conv_args={'channels': gat_channels,
                                               'xy_heads_ratio': 0.2,
                                               't_heads_ratio': 0.1,
                                               'iou_heads_ratio': 0.2,
                                               'act_att': F.leaky_relu,
                                               'act_gat': F.elu,
                                               'norm': GraphNorm,
                                               'dropout': 0.1,
                                               'bias': True,
                                               'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                                device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
fixations.shape, bboxes.shape

In [ ]:
graphs = grapher((ens, splits, fixations, bboxes)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

In [ ]:
from grapher import IOU_EuclideanTGazeGrapher

In [ ]:
grapher = IOU_EuclideanTGazeGrapher(gat_conv=CausalEuclideanSummaryIOU_TGATConv,
                                layers=layers,
                                output_gat_levels=output_gat_levels,
                                residual_gat_levels=residual_gat_levels,
                                att_heads=att_heads,
                                gat_conv_args={'channels': gat_channels,
                                               'xy_heads_ratio': 0.2,
                                               't_heads_ratio': 0.1,
                                               'iou_heads_ratio': 0.2,
                                               'act_att': F.leaky_relu,
                                               'act_gat': F.elu,
                                               'norm': GraphNorm,
                                               'dropout': 0.1,
                                               'bias': True,
                                               'w_init_f': lambda w: nn.init.xavier_uniform_(w)},
                                device=conf.device)
grapher

In [ ]:
expand_dims = nn.Conv1d(ns.shape[0], gat_channels, 1).to(conf.device)
ens = expand_dims(ns)
ens.shape

In [ ]:
fixations.shape, bboxes.shape

In [ ]:
graphs = grapher((ens, splits, fixations, bboxes)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

### Readout
<a id="readout"></a>

Since we are dealing with complete causal graphs --each node sees ALL of its predecessors and itself-- we developed a causal graph readout. All information is aggregated by the last node at each convolution. So, we take the last node of each graph selected by residual_gat_levels, concatenate them, and run them through linear layers, outputting the final predictions.

In [ ]:
from readout import CausalReadout

In [ ]:
sum([g.shape[0] for g in graphs])

In [ ]:
ro_channels=conf.ro_channels
ro_channels[0] = sum([g.shape[0] for g in graphs])

In [ ]:
causal_ro = CausalReadout(channels=ro_channels,
                          dropout=conf.ro_dropout,
                          device=conf.device)

In [ ]:
causal_ro

In [ ]:
predictions = causal_ro((graphs, splits))
predictions.shape, labels.shape

### Whole pipeline
<a id="full"></a>

Since we now have all individual components, we are ready to run the full model.

In [ ]:
from model.foveal_gat import FovealGAT

In [ ]:
gat_conv_args={'channels': gat_channels,
               'att_heads': 8,
               'act_att': F.leaky_relu,
               'act_gat': F.elu,
               'norm': GraphNorm,
               'dropout': 0.1,
               'bias': True,
               'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalGATConv,
                    gat_conv_args=gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
xyt_gat_conv_args={'channels': gat_channels,
                   'att_heads': 8,
                   'xy_heads_ratio': 0.2,
                   't_heads_ratio': 0.1,
                   'act_att': F.leaky_relu,
                   'act_gat': F.elu,
                   'norm': GraphNorm,
                   'dropout': 0.1,
                   'bias': True,
                   'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalXYTGATConv,
                    gat_conv_args=xyt_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalSummaryXYTGATConv,
                    gat_conv_args=xyt_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
euc_gat_conv_args={'channels': gat_channels,
                   'att_heads': 8,
                   'xy_heads_ratio': 0.2,
                   't_heads_ratio': 0.1,
                   'act_att': F.leaky_relu,
                   'act_gat': F.elu,
                   'norm': GraphNorm,
                   'dropout': 0.1,
                   'bias': True,
                   'w_init_f': lambda w: nn.init.xavier_uniform_(w)}

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalEuclideanTGATConv,
                    gat_conv_args=euc_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape

In [ ]:
model = FovealGAT(feat_extractor=feat_ext,
                    encoder=GazeSinCosPosEncoding(386, 16),
                    gat_conv=CausalEuclideanSummaryTGATConv,
                    gat_conv_args=euc_gat_conv_args,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape